# Solutions · 16 · Mixing and processing

Worked solutions to the exercises of [notebook 16](../notebooks/16_mixing_and_processing.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import flows, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

How does the mixing power of the xanthan drink scale with speed between 0.5 and 2 rev/s, compared with a Newtonian liquid? Estimate the exponent from the table.

In [2]:
car = (35.0, 0.002, 12.0, 0.25)
N = np.array([0.5, 2.0])
P = 70.0 * models.viscosity("carreau", 11.0 * N, *car) * N**2 * 0.1**3
exp_ = np.log(P[1] / P[0]) / np.log(N[1] / N[0])
n_loc = 1 + np.log(models.viscosity("carreau", 22.0, *car) / models.viscosity("carreau", 5.5, *car)) / np.log(4)
print(f"power ~ N^{exp_:.2f} (Newtonian laminar: N^2); local power-law index n = {n_loc:.2f} -> theory N^(n+1) = N^{n_loc + 1:.2f}")

power ~ N^1.25 (Newtonian laminar: N^2); local power-law index n = 0.25 -> theory N^(n+1) = N^1.25


In laminar mixing $P \propto \eta_{app}N^2$ with $\eta_{app} \propto N^{n-1}$, so $P \propto N^{n+1}$ - for this strongly shear-thinning drink
the power rises little more than linearly with speed, instead of quadratically.

## Exercise 2

Double the die gap to 2 mm at the same throughput. By what factor does the pressure drop fall, and how does that compare with a Newtonian melt (factor 8)?

In [3]:
K_loc, n_loc, W, L_die, Q = 3000 * 0.3 ** (0.3 - 1), 0.3, 1.0, 0.05, 100 / 3600 / 750
dp1 = flows.slit_power_law(K_loc, n_loc, W, 1e-3, Q=Q)["dp_per_L"] * L_die
dp2 = flows.slit_power_law(K_loc, n_loc, W, 2e-3, Q=Q)["dp_per_L"] * L_die
print(f"gap 1 mm: {dp1/1e5:.1f} bar, gap 2 mm: {dp2/1e5:.1f} bar -> factor {dp1/dp2:.2f} (power law 2^(2n+1) = {2**(2*n_loc+1):.2f}; Newtonian 8)")

gap 1 mm: 41.9 bar, gap 2 mm: 13.8 bar -> factor 3.03 (power law 2^(2n+1) = 3.03; Newtonian 8)


For a power-law fluid the die pressure scales as $H^{-(2n+1)}$ at fixed throughput: doubling the gap of this melt
cuts the pressure only about threefold, not eightfold as for a Newtonian liquid. (The local $K$ and $n$ used here
are those of the Carreau melt at high rates, where it behaves as a power law.)

## Exercise 3

**Implement it yourself:** write Tanner's swell formula yourself and tabulate the swell ratio against $N_1/\tau$ from 0 to 5. At what ratio does the extrudate swell by 50 %?

In [4]:
def tanner(ratio):
    return 0.1 + (1 + 0.5 * (ratio / 2) ** 2) ** (1 / 6)
ratios = np.linspace(0, 10, 101)
sw = tanner(ratios)
print(f"swell 1.5 is reached at N1/tau = {ratios[np.argmax(sw >= 1.5)]:.1f}; check library: {flows.tanner_swell(8.0, 1.0):.3f} vs {tanner(8.0):.3f}")

swell 1.5 is reached at N1/tau = 7.3; check library: 1.542 vs 1.542


Because of the 1/6 power, swell grows slowly: the normal stresses must be many times the shear stress before an
extrudate swells by half - which strongly elastic melts reach at high extrusion rates.